# DATA LOADING AND CLEANING

## Dataset for sentimental models

In this notebook we will load data about financial news from 2018 to 2023. Link of datasets: https://github.com/felixdrinkall/financial-news-dataset

In [1]:
# RUN THIS CELL IF USING COLAB

from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Ds"

Mounted at /content/drive


The data are provided in six separate JSON files, one for each year. In the following lines of code these files are concatenated into a single JSONL file. A copy of each dataset will be shared via Google Drive.

In [6]:
import os, json
import pandas as pd

#path = -MYPATH-                  IF NOT IN COLAB, PLEASE UNCOMMENT THIS LINE AND INSERT THE ACTUAL PATH WITH THE json FILES

def load_json_robust(path):
    #try with json lines
    try:
        return pd.read_json(path, lines=True)
    except ValueError:
        pass

    if path.endswith(".xz"):
        with lzma.open(path, "rt", encoding="utf-8") as f:
            text = f.read().strip()
    else:
    # fallback: standard json
      with open(path, "r", encoding="utf-8-sig") as f:
          text = f.read().strip()

    # empty rows or whitespaces
    if not text:
        raise ValueError(f"File vuoto: {path}")

    obj = json.loads(text)


    if isinstance(obj, list):
        return pd.DataFrame(obj)


    return pd.json_normalize(obj)


dfs = []
bad = []

for file in sorted(os.listdir(path)):
    if file.endswith(".json"):
        fp = os.path.join(path, file)
        print("Reading:", file)

        try:
            df = load_json_robust(fp)
            year = int(file.split("_")[0])
            df["year"] = year
            dfs.append(df)
        except Exception as e:
            bad.append((file, str(e)))
            print("Error", file, "->", e)

final_df = pd.concat(dfs, ignore_index=True)
print("Final shape:", final_df.shape)

if bad:
    print("\nFile with problems:")
    for b in bad:
        print(" -", b[0], ":", b[1])

final_df.head()

Reading: 2018_processed.json
Reading: 2019_processed.json
Reading: 2020_processed.json
Reading: 2021_processed.json
Reading: 2022_processed.json
Reading: 2023_processed.json
Final shape: (70571, 170)


,authors,date_download,date_modify,date_publish,description,filename,image_url,language,localpath,maintext,...,curr_day_price_CSCO,prev_day_price_NIO,next_day_price_NIO,curr_day_price_NIO,prev_day_price_MRNA,next_day_price_MRNA,curr_day_price_MRNA,prev_day_price_UNH,next_day_price_UNH,curr_day_price_UNH
0,[],2018-06-24 14:03:26+00:00,None,2018-06-22 18:39:00,Morgan Stanley is telling its clients to pay a...,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fmorga...,https://s.yimg.com/os/mit/media/p/common/image...,en,None,Morgan Stanley is telling its clients to pay a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,[],2018-11-15 21:39:58+00:00,None,2018-11-15 00:00:00,"The FCC voted to grant ""market access"" request...",https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Ffcc-o...,https://s.yimg.com/uu/api/res/1.2/qtgJvLeZydCO...,en,None,By David Shepardson\nWASHINGTON (Reuters - The...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,[],2018-12-31 02:00:12+00:00,None,2018-12-29 19:57:55,(Bloomberg) -- A lawsuit filed against Google ...,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fgoogl...,https://s.yimg.com/uu/api/res/1.2/7P7R2HFr_Cu0...,en,None,(Bloomberg) -- A lawsuit filed against Google ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,[],2018-09-13 00:56:51+00:00,None,2018-09-12 18:42:40,"With camera upgrades, better screens and more",https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Fapple...,https://s.yimg.com/uu/api/res/1.2/Jb5AstRxGBd3...,en,None,Apple on Wednesday unveiled a trio of new iPho...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,[],2018-04-25 02:02:32+00:00,None,2018-04-23 20:03:16,Google is reporting on Monday.,https%3A%2F%2Ffinance.yahoo.com%2Fnews%2Falpha...,https://s.yimg.com/uu/api/res/1.2/7rqX9riG7WX0...,en,None,"Google’s (GOOG, GOOGL) parent company Alphabet...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Alternatively, the data can be downloaded directly from Github by running the cell below.

In [ ]:
import os
import json
import requests
import pandas as pd
import lzma


repo_base_url = "https://raw.githubusercontent.com/FelixDrinkall/financial-news-dataset/main/data"
save_path = "./Data"
#save_path = -MYPATH-                  IF NOT IN COLAB, PLEASE UNCOMMENT THIS LINE AND INSERT THE CORRECT PATH
years = [2018, 2019, 2020, 2021, 2022, 2023]

os.makedirs(save_path, exist_ok=True)

# DOWNLOAD FILES FROM GITHUB
for year in years:
    filename = f"{year}_processed.json.xz"
    url = f"{repo_base_url}/{filename}"
    local_file = os.path.join(save_path, filename)

    if os.path.exists(local_file):
        print(f"Already exists: {filename}")
        continue

    print(f"Downloading: {filename}")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(local_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

print("\nDownload completed.\n")


# ROBUST JSON LOADING FUNCTION
def load_json_robust(path):
    try:
        return pd.read_json(path, lines=True, compression="infer")
    except ValueError:
        pass

    if path.endswith(".xz"):
        with lzma.open(path, "rt", encoding="utf-8") as f:
            text = f.read().strip()
    else:
        with open(path, "r", encoding="utf-8-sig") as f:
            text = f.read().strip()

    if not text:
        raise ValueError(f"Empty file: {path}")

    obj = json.loads(text)

    if isinstance(obj, list):
        return pd.DataFrame(obj)

    return pd.json_normalize(obj)

# READ + CONCATENATE
dfs = []
bad = []

for file in sorted(os.listdir(save_path)):
    if file.endswith(".json") or file.endswith(".json.xz"):
        fp = os.path.join(save_path, file)
        print("Reading:", file)

        try:
            df = load_json_robust(fp)
            year = int(file.split("_")[0])
            df["year"] = year
            dfs.append(df)
        except Exception as e:
            bad.append((file, str(e)))
            print("Error on", file, "->", e)

if dfs:
    final_df = pd.concat(dfs, ignore_index=True)
    print("\nFinal shape:", final_df.shape)
else:
    final_df = pd.DataFrame()
    print("\nNo dataframe was loaded.")

if bad:
    print("\nProblematic files:")
    for b in bad:
        print(" -", b[0], ":", b[1])

We select columns that are relevant for the exploratory data analysis and sentiment analysis:
- date_publish: timestamp when the article was originally published;
- description: summary of the article, this column will be the raw text input of the language models;
- maintext: full text content of the article;
- title: title of the article;
- mentioned_companies: list of stock ticker symbols of companies explicitly mentioned in the article;
- related_companies: list of stock ticker symbols of companies related to one of the mentioned companies, by being in the same industry;
- industries: list of industry codes associated with the article;
- sentiment: sentiment scores for negative, neutral, and positive sentiment;
- emotion: emotion analysis with probabilities for different emotions: neutral, surprise, disgust, joy, anger, sadness, and fear.

In [7]:
# Relevant columns
base_columns = [
    "date_publish",
    "description",
    "maintext",
    "title",
    "mentioned_companies",
    "related_companies",
    "industries",
    "sentiment",
    "emotion",
]

# Looking for missing columns in the concatenated dataframe
missing = [c for c in base_columns if c not in final_df.columns]
print("Missing columns:", missing)

# Selecting only relevant columns
base_df = final_df[base_columns].copy()

print("New shape:", base_df.shape)
print(base_df.head())

Missing columns: []
New shape: (70571, 9)
          date_publish                                        description  \
0  2018-06-22 18:39:00  Morgan Stanley is telling its clients to pay a...   
1  2018-11-15 00:00:00  The FCC voted to grant "market access" request...   
2  2018-12-29 19:57:55  (Bloomberg) -- A lawsuit filed against Google ...   
3  2018-09-12 18:42:40      With camera upgrades, better screens and more   
4  2018-04-23 20:03:16                     Google is reporting on Monday.   

                                            maintext  \
0  Morgan Stanley is telling its clients to pay a...   
1  By David Shepardson\nWASHINGTON (Reuters - The...   
2  (Bloomberg) -- A lawsuit filed against Google ...   
3  Apple on Wednesday unveiled a trio of new iPho...   
4  Google’s (GOOG, GOOGL) parent company Alphabet...   

                                               title  \
0  Morgan Stanley sees 'a pattern forming' of the...   
1  SpaceX, TeleSat Canada bids get U.S. nod to

In [8]:
print(base_df.isna().sum())

date_publish            0
description            24
maintext                0
title                   0
mentioned_companies     0
related_companies       0
industries              0
sentiment               0
emotion                 0
dtype: int64


In [9]:
base_df = base_df.dropna(subset=["description"])
print(base_df.isna().sum())

date_publish           0
description            0
maintext               0
title                  0
mentioned_companies    0
related_companies      0
industries             0
sentiment              0
emotion                0
dtype: int64


Before cleaning, the dataset contained 24 missing values in the description column, while all the other columns were complete.
After removing the rows with missing descriptions, the dataset no longer contains missing values in any column.

In [10]:
for col in base_df.columns:
    print(f"{col}:")
    print(base_df[col].apply(type).value_counts())
    print("------")

date_publish:
date_publish
<class 'str'>    70547
Name: count, dtype: int64
------
description:
description
<class 'str'>    70547
Name: count, dtype: int64
------
maintext:
maintext
<class 'str'>    70547
Name: count, dtype: int64
------
title:
title
<class 'str'>    70547
Name: count, dtype: int64
------
mentioned_companies:
mentioned_companies
<class 'list'>    70547
Name: count, dtype: int64
------
related_companies:
related_companies
<class 'list'>    70547
Name: count, dtype: int64
------
industries:
industries
<class 'list'>    70547
Name: count, dtype: int64
------
sentiment:
sentiment
<class 'dict'>    70547
Name: count, dtype: int64
------
emotion:
emotion
<class 'dict'>    70547
Name: count, dtype: int64
------


The column "mentioned_companies" contains the names of the companies mentioned in each financial article. In some rows, multiple companies are listed and stored as a string representation of a list. In order to extract all mentioned companies, these entries are converted into actual list objects.

In [11]:
import ast

base_df["mentioned_companies"] = base_df["mentioned_companies"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

all_companies = base_df["mentioned_companies"].explode().unique()
print(all_companies)

['GOOGL' 'TSLA' 'MU' 'AAPL' 'INTC' 'META' 'T' 'V' 'AMZN' 'WMT' 'NFLX'
 'JNJ' 'C' 'BA' 'MSFT' 'DIS' 'VZ' 'MA' 'GS' 'GE' 'KO' 'QCOM' 'NVDA' 'ADBE'
 'BAC' 'WFC' 'BABA' 'JPM' 'CRM' 'BRK' 'SQ' 'PYPL' 'PFE' 'XOM' 'CVX' 'ROKU'
 'ORCL' 'SHOP' 'COST' 'CMCSA' 'AVGO' 'HD' 'PG' 'LLY' 'MRK' 'CSCO' 'NIO'
 'MRNA' 'UNH']


Example of mentioned company and main article:

In [12]:
i = 45
print(base_df["mentioned_companies"][i])
print(base_df["maintext"][i])

['TSLA']
SpaceX is gearing up in its effort to build a rocket that could carry mankind to Mars and beyond. This week, the company posted its first job posting dedicated to the project.
As part of its grand vision, SpaceX is designing the Big Falcon Rocket, or BFR, as a launch vehicle capable of carrying people to Mars. CEO Elon Musk envisions that the technology will eventually replace the Falcon 9 and even shuttle passengers between New York and Los Angeles in about 25 minutes.
To achieve this, SpaceX is hiring a “build engineer” with experience in aerospace and mechanical engineering who can “work long hours and weekends” whenever needed. “Working directly in the Vehicle Engineering group, the goal of this team is to investigate, test, and develop new hardware, software, and automation efforts capable of supporting advanced metallic and composite joining methods for the BFR,” said the job description, which listed no salary range or application deadline.
Announced in early 2017, the 

In [13]:
for i in range(5):
    print(base_df["sentiment"][i])

{'negative': 0.0005622319295071065, 'neutral': 0.006970268674194813, 'positive': 0.99246746301651}
{'negative': 0.00028582950471900403, 'neutral': 0.0016880716430023313, 'positive': 0.9980260729789734}
{'negative': 0.23072107136249542, 'neutral': 0.36165717244148254, 'positive': 0.407621830701828}
{'negative': 0.005342656280845404, 'neutral': 0.16495321691036224, 'positive': 0.8297041654586792}
{'negative': 0.00031546535319648683, 'neutral': 5.8617933973437175e-05, 'positive': 0.9996259212493896}


Each instance in the "sentiment" column is represented as a dictionary, where the three labels "Positive", "Negative" and "Neutral" are each associated with a probability. The sentiment assigned to the article corresponds to the label with the highest probability.

In [14]:
base_df["sentiment_label"] = base_df["sentiment"].apply(
    lambda x: max(x.items(), key=lambda item: item[1])[0]
)

print(base_df["sentiment_label"].value_counts())

sentiment_label
positive    39098
neutral     19028
negative    12421
Name: count, dtype: int64


We can save the final dataset as a JSONL file. A copy will be shared in Google Drive, so the following code may be skipped.

In [ ]:
base_df.to_json(
    "/content/drive/MyDrive/Ds/financial_news_base.jsonl",
    orient="records",
    lines=True
)

If the models are run locally and not on Colab, it is necessary to save the file in this way. Additionally, if you want to use the dashboard, the dataset must be saved locally so that the subsequent section works correctly.

In [ ]:
base_df.to_json(
    "financial_news_base.jsonl",
    orient="records",
    lines=True
)

print("Dataset saved to financial_news_base.jsonl")

## Dataset for Dashboard

The dashboard uses the same data as the models, but they will be saved in **CSV format** to make the loading process faster.

In [3]:
file_path = "financial_news_base.jsonl"

base_df = pd.read_json(file_path, lines=True)

base_df.head()

,date_publish,description,maintext,title,mentioned_companies,related_companies,industries,sentiment,emotion,sentiment_label
0,2018-06-22 18:39:00,Morgan Stanley is telling its clients to pay a...,Morgan Stanley is telling its clients to pay a...,Morgan Stanley sees 'a pattern forming' of the...,[GOOGL],"[IGLD, RAMP, NSR, TWTR, ACXM, COR, PINS, META,...",[7375],"{'negative': 0.0005622319, 'neutral': 0.006970...","{'neutral': 0.9212942719, 'surprise': 0.039100...",positive
1,2018-11-15 00:00:00,"The FCC voted to grant ""market access"" request...",By David Shepardson\nWASHINGTON (Reuters - The...,"SpaceX, TeleSat Canada bids get U.S. nod to ex...","[TSLA, MU]","[IKGH, CLLS, EBIO, GMAB, GBS, WIMI, WALD, FGEN...","[3674, 9999]","{'negative': 0.0002858295, 'neutral': 0.001688...","{'neutral': 0.8668617606000001, 'joy': 0.05769...",positive
2,2018-12-29 19:57:55,(Bloomberg) -- A lawsuit filed against Google ...,(Bloomberg) -- A lawsuit filed against Google ...,Google Wins Dismissal of Suit Over Facial Reco...,[GOOGL],"[IGLD, RAMP, NSR, TWTR, ACXM, COR, PINS, META,...",[7375],"{'negative': 0.2307210714, 'neutral': 0.361657...","{'anger': 0.7141016126, 'neutral': 0.202039063...",positive
3,2018-09-12 18:42:40,"With camera upgrades, better screens and more",Apple on Wednesday unveiled a trio of new iPho...,"Apple Unveils iPhone XS, iPhone XS Max and Che...",[AAPL],"[None, TDC, HPQ, SCKT, OMCL, CRAY, ZEPP, IBM, ...",[3571],"{'negative': 0.0053426563, 'neutral': 0.164953...","{'neutral': 0.9592552185000001, 'surprise': 0....",positive
4,2018-04-23 20:03:16,Google is reporting on Monday.,"Google’s (GOOG, GOOGL) parent company Alphabet...",Alphabet Q1 2018 earnings,"[GOOGL, INTC, META, AAPL]","[IGLD, TDC, NSAT, CRAY, TCX, FLEX, SMCI, WBMD,...","[3571, 3679, 7375]","{'negative': 0.0003154654, 'neutral': 5.86179e...","{'neutral': 0.6759750247, 'surprise': 0.166648...",positive


The created datasets are saved in the specified folders so that the dashboard knows where to retrieve the data.

In [4]:
import os

# Define the folder where datasets will be saved
output_folder = r".\Dashboard\data\dashboard"

# Create the folder if it does not exist
os.makedirs(output_folder, exist_ok=True)

# Define output file paths
full_dataset_path = os.path.join(output_folder, "full_dataset.csv")
balanced_dataset_path = os.path.join(output_folder, "balanced_dataset.csv")

The choice of the data filtering system will be explained within the notebooks and the dashboard.

In [5]:
from transformers import DistilBertTokenizer as _TempTokenizer

MAX_LEN = 256

# Step 1: Filter rows with more than 256 tokens on the ORIGINAL dataset (before balancing)
_tokenizer_temp = _TempTokenizer.from_pretrained("distilbert-base-uncased")
mask_256 = base_df["description"].apply(
    lambda t: len(_tokenizer_temp.encode(str(t), add_special_tokens=True)) <= MAX_LEN
)
df_filtered = base_df[mask_256].reset_index(drop=True)

# Step 2: Balance classes AFTER filtering
target_per_class = 25000 // 3

dfs = []
for label in df_filtered["sentiment_label"].unique():
    subset = df_filtered[df_filtered["sentiment_label"] == label]
    dfs.append(subset.sample(n=target_per_class, random_state=100))

df_balanced = pd.concat(dfs).sample(frac=1, random_state=100).reset_index(drop=True)

c:\Users\Matti\Desktop\Text_m\VT\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Token indices sequence length is longer than the specified maximum sequence length for this model (1534 > 512). Running this sequence through the model will result in indexing errors


The column indicating the sectors is translated from SIC codes into macro-sectors to make the graphical representations within the dashboard more intuitive.

In [6]:
# Function to convert a SIC code into a macro sector
def sic_to_sector(code):
    if pd.isna(code):
        return None

    code = int(float(code))
    prefix = int(str(code)[:2])

    if 20 <= prefix <= 39:
        return "Manufacturing"
    elif 40 <= prefix <= 49:
        return "Transportation & Communications"
    elif 50 <= prefix <= 59:
        return "Wholesale & Retail"
    elif 60 <= prefix <= 67:
        return "Finance"
    elif 70 <= prefix <= 89:
        return "Services"
    else:
        return "Other"

# Function to convert a list of SIC codes into sector groups
def convert_industries(ind_list):
    if not isinstance(ind_list, list):
        return None

    sectors = set()

    for code in ind_list:
        if pd.notna(code):
            sector = sic_to_sector(code)
            if sector is not None:
                sectors.add(sector)

    return list(sectors) if sectors else None

In [7]:
# Add sector_group column to both datasets
base_df["sector_group"] = base_df["industries"].apply(convert_industries)
df_balanced["sector_group"] = df_balanced["industries"].apply(convert_industries)

print("Sector group column added to both datasets.")

Sector group column added to both datasets.


The two datasets are saved in their respective folders.


In [8]:
# Save the full dataset
base_df.to_csv(full_dataset_path, index=False)

# Save the balanced dataset
df_balanced.to_csv(balanced_dataset_path, index=False)

print("Datasets saved successfully.")
print("Full dataset:", full_dataset_path)
print("Balanced dataset:", balanced_dataset_path)

Datasets saved successfully.
Full dataset: .\Dashboard\data\dashboard\full_dataset.csv
Balanced dataset: .\Dashboard\data\dashboard\balanced_dataset.csv
